# 03 — Compute breakpoints (BIC sweep, multiple trials)

Adapted from `legacy/notebooks/02_breakpoints.ipynb`.

For each well listed in `results/chosen_method.csv`, this notebook:
  1. Loads the chosen smoothed file.
  2. Runs `best_n_breakpoints` over `max_breakpoints` × `n_trials`.
  3. Saves the full result as JSON in `data/breakpoints/2022_02/`.
  4. Records each run in `results/runs.csv` via `register_run`.

`piecewise_regression` uses a randomly initialised optimiser, so multiple
trials are needed to assess agreement on the chosen N.

This notebook is intentionally NOT fully automated — review the JSON output
in notebook 04 before committing to a final N.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, json, logging
if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from karst_analysis.runs import register_run
from karst_analysis.sec.breakpoints import best_n_breakpoints

logging.basicConfig(level=logging.WARNING, format="%(message)s")

In [2]:
# Parameters
MAX_BREAKPOINTS = 10
N_TRIALS        = 3
TOLERANCE       = 1e-5
MIN_DISTANCE    = 0.01

USE_LOG10       = True   # operate on log10 of conductivity if True

CHOICES_CSV     = Path("results/chosen_method.csv")
BP_DIR          = Path("data/breakpoints/2022_02")
BP_DIR.mkdir(parents=True, exist_ok=True)

choices = pd.read_csv(CHOICES_CSV)
choices

FileNotFoundError: [Errno 2] No such file or directory: 'results\\chosen_method.csv'

In [ ]:
# Helper: serialise a result dict to JSON-safe form
def _serialise(results):
    out = {}
    for trial_name, info in results.items():
        df = info["df"].copy()
        # Convert estimates dicts to a JSON-safe form
        if "estimates" in df.columns:
            df["estimates"] = df["estimates"].apply(lambda x: x if x is None else dict(x))
        out[trial_name] = {
            "df": df.to_dict(),
            "best_n_breakpoint_bic": int(info["best_n_breakpoint_bic"]),
            "min_bic_n_breakpoint":  int(info["min_bic_n_breakpoint"]),
            "best_n_breakpoint_rss": int(info["best_n_breakpoint_rss"]),
        }
    return out

In [ ]:
# Run BIC sweep for every chosen well
for _, row in tqdm(choices.iterrows(), total=len(choices), desc="Wells"):
    well_id = row["well_id"]
    date    = str(row["date"])
    method  = row["method"]
    csv_path = Path(row["chosen_file"])

    df = pd.read_csv(csv_path)

    # Pick X (depth) and Y (conductivity, possibly log10)
    x_col = "depth_m"
    y_col = "log10_sec_uS_cm" if USE_LOG10 and "log10_sec_uS_cm" in df.columns else "sec_uS_cm"

    x = df[x_col].to_numpy()
    y = df[y_col].dropna().to_numpy()
    x = df[x_col].iloc[df[y_col].notna().values].to_numpy()  # align with non-NaN y

    params = {
        "method": "breakpoints",
        "max_breakpoints": MAX_BREAKPOINTS,
        "n_trials": N_TRIALS,
        "tolerance": TOLERANCE,
        "min_distance": MIN_DISTANCE,
        "smoothing_method": method,
        "y_space": "log10" if y_col.startswith("log10") else "linear",
    }

    with register_run(
        stage="breakpoints",
        well_id=well_id, date=date,
        input_file=str(csv_path), params=params,
        output_dir=BP_DIR, extension="json",
    ) as run:
        results = best_n_breakpoints(
            x, y,
            max_breakpoints=MAX_BREAKPOINTS,
            n_trials=N_TRIALS,
            tolerance=TOLERANCE,
            min_distance=MIN_DISTANCE,
        )
        # Print per-trial summary
        for tname, info in results.items():
            print(f"  {well_id} {tname}: "
                  f"elbow_bic={info['best_n_breakpoint_bic']}  "
                  f"min_bic_n={info['min_bic_n_breakpoint']}  "
                  f"elbow_rss={info['best_n_breakpoint_rss']}")

        # Persist as JSON
        with open(run.output_path, "w") as f:
            json.dump(_serialise(results), f, default=str, indent=2)

        run.note = ""
        print(f"  → {run.output_path.name}")

print("\nDone. Move on to notebook 04 to evaluate.")